# NRC-VAD Arousal Intensity

This notebook estimates annual affective intensity for ADHD, Autism, and the three baseline terms. It reuses the frame-aware NRC-VAD collocate handoff built in the Sentiment notebook and computes Baes-style annual arousal indices from local target-window collocates.


## Setup

The diachronic axis is publication year (`lsc_year`). Target estimates are reported for the substantive core aggregate and for clinical-only, lived-only, and mixed frame strata. Baseline terms remain unframed comparator series.


In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
VAD_MATCH_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_collocate_matches.parquet"
VAD_COVERAGE_PATH = PROJECT_ROOT / "data/interim/lsc/vad/lsc_vad_context_coverage.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/lsc/intensity"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/intensity"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

ANNUAL_AROUSAL_PATH = OUTPUT_DIR / "lsc_intensity_annual_arousal.csv"
COVERAGE_PATH = OUTPUT_DIR / "lsc_intensity_coverage.csv"
TOP_COLLOCATES_PATH = OUTPUT_DIR / "lsc_intensity_top_collocates.csv"
AUDIT_FLAGS_PATH = OUTPUT_DIR / "lsc_intensity_audit_flags.csv"
TREND_SUMMARY_PATH = OUTPUT_DIR / "lsc_intensity_trend_models.csv"
TRAJECTORY_PLOT_PATH = FIGURE_DIR / "lsc_intensity_arousal_trajectories.png"

EXPECTED_YEARS = list(range(2014, 2027))
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_UNITS = TARGET_UNITS + BASELINE_UNITS
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BASELINE_FRAME_STRATUM = "unframed_baseline"
BOOTSTRAP_REPETITIONS = 500
RANDOM_SEED = 123
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75

UNIT_COLOURS = {
    "ADHD": "#2F6F9F",
    "Autism": "#B66A4A",
    "frustration": "#4F8F78",
    "loneliness": "#7FA68A",
    "sadness": "#9AA6A1",
}
UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
    "mixed": "Mixed clinical/lived framing",
    "unframed_baseline": "Comparator term",
}
FRAME_COLORS = {
    "substantive_core_overall": "#263238",
    "clinical_only": "#4F8DB3",
    "lived_only": "#C98263",
    "mixed": "#79A889",
    "substantive_other": "#A998C9",
    "non_substantive_or_insufficient": "#B8C0C5",
    "unframed_baseline": "#7B8785",
}
CONDITION_FRAME_COLORS = {
    "ADHD": {
        "substantive_core_overall": "#2F6F9F",
        "clinical_only": "#75A9C8",
        "lived_only": "#AECFE0",
        "mixed": "#D4E4EC",
    },
    "Autism": {
        "substantive_core_overall": "#B66A4A",
        "clinical_only": "#CE8D70",
        "lived_only": "#E1B49D",
        "mixed": "#F2D8CF",
    },
}
FRAME_MARKERS = {
    "substantive_core_overall": "o",
    "clinical_only": "s",
    "lived_only": "^",
    "mixed": "D",
    "unframed_baseline": "o",
}
LSC_FIGURE_DPI = 300


## Load VAD Handoff

The handoff contains one row per matched collocate occurrence and one coverage row per analysis context. It is produced by the Sentiment notebook, so the tokenisation, lemmatisation, focal-term exclusion, MWE matching, and frame-stratum contract remain identical for Sentiment and Intensity.


In [2]:
if not VAD_MATCH_PATH.exists():
    raise FileNotFoundError(f"Missing VAD match handoff: {VAD_MATCH_PATH}")
if not VAD_COVERAGE_PATH.exists():
    raise FileNotFoundError(f"Missing VAD coverage handoff: {VAD_COVERAGE_PATH}")

vad_matches = pd.read_parquet(VAD_MATCH_PATH)
context_coverage = pd.read_parquet(VAD_COVERAGE_PATH)

required_match_columns = {
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "frame_stratum",
    "registered_domain",
    "collocate",
    "valence",
    "arousal",
    "dominance",
}
required_coverage_columns = {
    "context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "candidate_collocate_tokens",
    "matched_vad_units",
    "matched_token_positions",
    "has_vad_match",
}
missing_match_columns = sorted(required_match_columns - set(vad_matches.columns))
missing_coverage_columns = sorted(required_coverage_columns - set(context_coverage.columns))
if missing_match_columns:
    raise RuntimeError(f"VAD match handoff is missing columns: {missing_match_columns}")
if missing_coverage_columns:
    raise RuntimeError(f"VAD coverage handoff is missing columns: {missing_coverage_columns}")

observed_units = sorted(context_coverage["analysis_unit"].dropna().unique())
observed_years = sorted(context_coverage["lsc_year"].dropna().astype(int).unique())
observed_frame_strata = sorted(context_coverage["frame_stratum"].dropna().unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
missing_target_strata = sorted(set(TARGET_FRAME_STRATA) - set(observed_frame_strata))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units in VAD coverage handoff: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years in VAD coverage handoff: {missing_years}")
if missing_target_strata:
    raise RuntimeError(f"Missing expected target frame strata in VAD coverage handoff: {missing_target_strata}")

handoff_summary = pd.DataFrame(
    {
        "metric": ["matched_vad_collocate_rows", "context_rows", "documents", "analysis_units", "frame_strata", "years"],
        "value": [
            len(vad_matches),
            len(context_coverage),
            context_coverage["doc_id"].nunique(),
            ", ".join(observed_units),
            ", ".join(observed_frame_strata),
            f"{min(observed_years)}-{max(observed_years)}",
        ],
    }
)
handoff_summary


,metric,value
0,matched_vad_collocate_rows,1069580
1,context_rows,311030
2,documents,192046
3,analysis_units,"ADHD, Autism, frustration, loneliness, sadness"
4,frame_strata,"clinical_only, lived_only, mixed, substantive_core_overall, unframed_baseline"
5,years,2014-2026


## Annual Arousal Index

Annual arousal is the weighted mean of all matched NRC-VAD collocate occurrences for each analysis unit, publication year, and frame stratum. Coverage and small-cell flags are carried alongside the index.


In [3]:
GROUP_COLUMNS = ["lsc_year", "analysis_unit", "frame_stratum"]

coverage = (
    context_coverage.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        context_rows=("context_row_id", "size"),
        documents=("doc_id", "nunique"),
        candidate_collocate_tokens=("candidate_collocate_tokens", "sum"),
        matched_vad_units_coverage=("matched_vad_units", "sum"),
        matched_token_positions=("matched_token_positions", "sum"),
        contexts_with_vad_match=("has_vad_match", "sum"),
    )
)
coverage["matched_token_coverage"] = coverage["matched_token_positions"] / coverage["candidate_collocate_tokens"].replace(0, np.nan)
coverage["context_match_coverage"] = coverage["contexts_with_vad_match"] / coverage["context_rows"].replace(0, np.nan)
coverage["small_cell_flag"] = coverage["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    coverage["context_rows"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | coverage["documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)

annual_arousal = (
    vad_matches.groupby([*GROUP_COLUMNS, "term_role", "target_group"], as_index=False)
    .agg(
        arousal_mean=("arousal", "mean"),
        arousal_sd=("arousal", "std"),
        valence_mean_for_reference=("valence", "mean"),
        dominance_mean_for_reference=("dominance", "mean"),
        matched_vad_units=("arousal", "size"),
        unique_collocates=("collocate", "nunique"),
        documents_with_matches=("doc_id", "nunique"),
    )
)
annual_arousal = annual_arousal.merge(coverage, on=GROUP_COLUMNS, how="left")
annual_arousal = annual_arousal.sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_arousal.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,arousal_mean,arousal_sd,valence_mean_for_reference,dominance_mean_for_reference,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,0.028210,0.342806,0.061856,0.041583,4317,1157,791,1286,808,4888,4317,4443,1249,0.908961,0.971229,False
1,2015,ADHD,clinical_only,target,ADHD,0.020463,0.338259,0.074820,0.045729,3869,1119,763,1203,774,4464,3869,3979,1171,0.891353,0.973400,False
2,2016,ADHD,clinical_only,target,ADHD,0.031395,0.339819,0.038802,0.028800,3833,1112,784,1149,790,4295,3833,3933,1124,0.915716,0.978242,False
3,2017,ADHD,clinical_only,target,ADHD,0.036331,0.343705,0.039036,0.035293,3594,1107,735,1099,751,4123,3594,3709,1063,0.899588,0.967243,False
4,2018,ADHD,clinical_only,target,ADHD,0.034197,0.345063,0.039881,0.024145,4089,1154,782,1182,787,4602,4089,4193,1164,0.911126,0.984772,False
5,2019,ADHD,clinical_only,target,ADHD,0.038305,0.346618,0.010419,0.015428,3475,1004,664,1000,674,3957,3475,3571,981,0.902451,0.981000,False
6,2020,ADHD,clinical_only,target,ADHD,0.040228,0.340137,0.031440,0.025447,3136,949,662,959,668,3590,3136,3237,932,0.901671,0.971846,False
7,2021,ADHD,clinical_only,target,ADHD,0.034450,0.353351,0.025841,0.010944,3177,1033,634,928,636,3565,3177,3272,911,0.917812,0.981681,False
8,2022,ADHD,clinical_only,target,ADHD,0.015353,0.340910,0.059404,0.042366,3202,992,624,930,634,3606,3202,3305,906,0.916528,0.974194,False
9,2023,ADHD,clinical_only,target,ADHD,0.039470,0.343197,0.058146,0.056264,2548,861,470,750,477,2845,2548,2598,729,0.913181,0.972000,False


## Bootstrap Confidence Intervals

The bootstrap resamples documents within each unit-year-frame stratum. This matches the Sentiment uncertainty contract and avoids treating collocates from the same document as independent observations.


In [4]:
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_records: list[dict[str, object]] = []

doc_scores = (
    vad_matches.groupby([*GROUP_COLUMNS, "doc_id"], as_index=False)
    .agg(arousal_sum=("arousal", "sum"), matched_vad_units=("arousal", "size"))
)

for group_values, frame in doc_scores.groupby(GROUP_COLUMNS, sort=True):
    arousal_sums = frame["arousal_sum"].to_numpy(dtype=float)
    unit_counts = frame["matched_vad_units"].to_numpy(dtype=float)
    n_docs = len(frame)
    estimates = np.empty(BOOTSTRAP_REPETITIONS, dtype=float)
    for _ in range(BOOTSTRAP_REPETITIONS):
        sample_index = rng.integers(0, n_docs, size=n_docs)
        denominator = unit_counts[sample_index].sum()
        estimates[_] = arousal_sums[sample_index].sum() / denominator if denominator else np.nan
    bootstrap_records.append(
        {
            "lsc_year": int(group_values[0]),
            "analysis_unit": group_values[1],
            "frame_stratum": group_values[2],
            "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
            "bootstrap_unit": "doc_id",
            "arousal_bootstrap_mean": float(np.nanmean(estimates)),
            "arousal_ci_low": float(np.nanquantile(estimates, 0.025)),
            "arousal_ci_high": float(np.nanquantile(estimates, 0.975)),
        }
    )

bootstrap_arousal = pd.DataFrame(bootstrap_records)
annual_arousal = annual_arousal.merge(bootstrap_arousal, on=GROUP_COLUMNS, how="left")
annual_arousal.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,arousal_mean,arousal_sd,valence_mean_for_reference,dominance_mean_for_reference,matched_vad_units,unique_collocates,documents_with_matches,context_rows,documents,candidate_collocate_tokens,matched_vad_units_coverage,matched_token_positions,contexts_with_vad_match,matched_token_coverage,context_match_coverage,small_cell_flag,bootstrap_repetitions,bootstrap_unit,arousal_bootstrap_mean,arousal_ci_low,arousal_ci_high
0,2014,ADHD,clinical_only,target,ADHD,0.028210,0.342806,0.061856,0.041583,4317,1157,791,1286,808,4888,4317,4443,1249,0.908961,0.971229,False,500,doc_id,0.028454,0.018364,0.039730
1,2015,ADHD,clinical_only,target,ADHD,0.020463,0.338259,0.074820,0.045729,3869,1119,763,1203,774,4464,3869,3979,1171,0.891353,0.973400,False,500,doc_id,0.020513,0.008863,0.033627
2,2016,ADHD,clinical_only,target,ADHD,0.031395,0.339819,0.038802,0.028800,3833,1112,784,1149,790,4295,3833,3933,1124,0.915716,0.978242,False,500,doc_id,0.031881,0.019174,0.045066
3,2017,ADHD,clinical_only,target,ADHD,0.036331,0.343705,0.039036,0.035293,3594,1107,735,1099,751,4123,3594,3709,1063,0.899588,0.967243,False,500,doc_id,0.036727,0.023307,0.049617
4,2018,ADHD,clinical_only,target,ADHD,0.034197,0.345063,0.039881,0.024145,4089,1154,782,1182,787,4602,4089,4193,1164,0.911126,0.984772,False,500,doc_id,0.034036,0.022804,0.047250
5,2019,ADHD,clinical_only,target,ADHD,0.038305,0.346618,0.010419,0.015428,3475,1004,664,1000,674,3957,3475,3571,981,0.902451,0.981000,False,500,doc_id,0.038916,0.025259,0.052495
6,2020,ADHD,clinical_only,target,ADHD,0.040228,0.340137,0.031440,0.025447,3136,949,662,959,668,3590,3136,3237,932,0.901671,0.971846,False,500,doc_id,0.040291,0.027727,0.053453
7,2021,ADHD,clinical_only,target,ADHD,0.034450,0.353351,0.025841,0.010944,3177,1033,634,928,636,3565,3177,3272,911,0.917812,0.981681,False,500,doc_id,0.034667,0.021331,0.049226
8,2022,ADHD,clinical_only,target,ADHD,0.015353,0.340910,0.059404,0.042366,3202,992,624,930,634,3606,3202,3305,906,0.916528,0.974194,False,500,doc_id,0.015562,0.002680,0.028980
9,2023,ADHD,clinical_only,target,ADHD,0.039470,0.343197,0.058146,0.056264,2548,861,470,750,477,2845,2548,2598,729,0.913181,0.972000,False,500,doc_id,0.039646,0.026014,0.053214


## Trend Models

The main descriptive trend is OLS arousal-on-centred-year for each reported series. Residual autocorrelation is flagged with a Durbin-Watson diagnostic; when flagged, an AR(1)-transformed sensitivity slope is reported in the trend table.


In [5]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
for group_values, frame in annual_arousal.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    trend_rows.append(
        {
            "analysis_unit": analysis_unit,
            "frame_stratum": frame_stratum,
            "term_role": term_role,
            "target_group": target_group,
            "index_name": "arousal_mean",
            **fit_trend(frame, "arousal_mean"),
        }
    )
trend_summary = pd.DataFrame(trend_rows)
trend_summary.to_csv(TREND_SUMMARY_PATH, index=False)
trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,arousal_mean,13,2020.0,0.031835,0.000212,0.000619,0.738479,0.010548,-0.079403,0.102702,2.153049,-0.166269,False,NaN,NaN,-0.021188,0.058214
1,ADHD,lived_only,target,ADHD,arousal_mean,13,2020.0,0.000273,0.001496,0.000896,0.123193,0.202165,0.129635,0.449628,1.779185,0.073694,False,NaN,NaN,0.064579,-0.065056
2,ADHD,mixed,target,ADHD,arousal_mean,13,2020.0,0.006381,-0.000152,0.001328,0.910699,0.001196,-0.089604,-0.034582,1.583851,0.190696,False,NaN,NaN,0.218413,0.308017
3,ADHD,substantive_core_overall,target,ADHD,arousal_mean,13,2020.0,0.022916,0.000065,0.000506,0.899570,0.001514,-0.089257,0.038911,1.469599,0.245362,False,NaN,NaN,0.076607,0.165864
4,Autism,clinical_only,target,Autism,arousal_mean,13,2020.0,0.030247,0.000986,0.000443,0.047890,0.310496,0.247814,0.557221,1.469940,-0.071865,False,NaN,NaN,0.458649,0.210836
5,Autism,lived_only,target,Autism,arousal_mean,13,2020.0,-0.016087,0.002541,0.002100,0.251506,0.117528,0.037303,0.342823,2.412959,-0.240680,False,NaN,NaN,0.053823,0.016520
6,Autism,mixed,target,Autism,arousal_mean,13,2020.0,-0.008827,0.000427,0.000969,0.667879,0.017359,-0.071972,0.131754,1.883419,-0.224519,False,NaN,NaN,-0.113415,-0.041443
7,Autism,substantive_core_overall,target,Autism,arousal_mean,13,2020.0,0.009681,0.001092,0.000846,0.222925,0.131713,0.052778,0.362923,2.535658,-0.410263,False,NaN,NaN,-0.027957,-0.080735
8,frustration,unframed_baseline,baseline,baseline,arousal_mean,13,2020.0,0.020226,-0.000443,0.000560,0.446137,0.053710,-0.032316,-0.231755,0.644913,0.587777,True,-0.00224,0.077991,0.436881,0.469198
9,loneliness,unframed_baseline,baseline,baseline,arousal_mean,13,2020.0,0.022621,-0.000604,0.000578,0.318524,0.090281,0.007579,-0.300468,1.818406,0.056028,False,NaN,NaN,0.064202,0.056622


## Diagnostics And Save Tables

Diagnostics flag sparse frame-year cells, low VAD coverage, high collocate concentration, and trend-series residual autocorrelation. They are interpretive warnings rather than exclusion rules.


In [6]:
collocate_counts = (
    vad_matches.groupby(["lsc_year", "analysis_unit", "frame_stratum", "collocate"], as_index=False)
    .agg(
        occurrences=("collocate", "size"),
        arousal=("arousal", "mean"),
        valence=("valence", "mean"),
        documents=("doc_id", "nunique"),
    )
)
collocate_totals = collocate_counts.groupby(["lsc_year", "analysis_unit", "frame_stratum"])["occurrences"].transform("sum")
collocate_counts["occurrence_share"] = collocate_counts["occurrences"] / collocate_totals
collocate_counts["weighted_arousal_contribution"] = collocate_counts["occurrences"] * collocate_counts["arousal"]

collocate_counts["total_matches_for_unit_year_frame"] = collocate_totals
top_collocates = (
    collocate_counts.sort_values(["lsc_year", "analysis_unit", "frame_stratum", "occurrences"], ascending=[True, True, True, False])
    .groupby(["lsc_year", "analysis_unit", "frame_stratum"], as_index=False)
    .head(15)
    .reset_index(drop=True)
)

concentration = (
    collocate_counts.sort_values(["lsc_year", "analysis_unit", "frame_stratum", "occurrences"], ascending=[True, True, True, False])
    .groupby(["lsc_year", "analysis_unit", "frame_stratum"], as_index=False)
    .head(5)
    .groupby(["lsc_year", "analysis_unit", "frame_stratum"], as_index=False)
    .agg(top5_collocate_share=("occurrence_share", "sum"))
)

coverage_for_flags = annual_arousal.merge(concentration, on=GROUP_COLUMNS, how="left")
flag_rows = []
for row in coverage_for_flags.itertuples(index=False):
    flags = []
    if bool(row.small_cell_flag):
        flags.append("small_frame_year_cell")
    if row.context_match_coverage < 0.90:
        flags.append("low_context_vad_match_coverage_lt_0_90")
    if row.matched_token_coverage < 0.70:
        flags.append("low_token_vad_coverage_lt_0_70")
    if pd.notna(row.top5_collocate_share) and row.top5_collocate_share > 0.35:
        flags.append("top5_collocate_share_gt_0_35")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "flags": ";".join(flags),
                "context_rows": row.context_rows,
                "documents": row.documents,
                "context_match_coverage": row.context_match_coverage,
                "matched_token_coverage": row.matched_token_coverage,
                "top5_collocate_share": row.top5_collocate_share,
            }
        )

for row in trend_summary.loc[trend_summary["autocorrelation_flag"]].itertuples(index=False):
    flag_rows.append(
        {
            "lsc_year": pd.NA,
            "analysis_unit": row.analysis_unit,
            "frame_stratum": row.frame_stratum,
            "flags": "trend_residual_autocorrelation",
            "context_rows": pd.NA,
            "documents": pd.NA,
            "context_match_coverage": pd.NA,
            "matched_token_coverage": pd.NA,
            "top5_collocate_share": pd.NA,
        }
    )

audit_flag_columns = [
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "flags",
    "context_rows",
    "documents",
    "context_match_coverage",
    "matched_token_coverage",
    "top5_collocate_share",
]
audit_flags = pd.DataFrame(flag_rows, columns=audit_flag_columns)

annual_arousal.to_csv(ANNUAL_AROUSAL_PATH, index=False)
coverage_for_flags.to_csv(COVERAGE_PATH, index=False)
top_collocates.to_csv(TOP_COLLOCATES_PATH, index=False)
audit_flags.to_csv(AUDIT_FLAGS_PATH, index=False)

pd.DataFrame(
    {
        "output": ["annual_arousal", "coverage", "top_collocates", "trend_summary", "audit_flags"],
        "path": [ANNUAL_AROUSAL_PATH, COVERAGE_PATH, TOP_COLLOCATES_PATH, TREND_SUMMARY_PATH, AUDIT_FLAGS_PATH],
        "rows": [len(annual_arousal), len(coverage_for_flags), len(top_collocates), len(trend_summary), len(audit_flags)],
    }
)


,output,path,rows
0,annual_arousal,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_annual_arousal.csv,143
1,coverage,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_coverage.csv,143
2,top_collocates,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_top_collocates.csv,2145
3,trend_summary,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_trend_models.csv,11
4,audit_flags,/Users/jakoblutkemeier/Documents/msc-nlp-therapy-speak/data/processed/lsc/intensity/lsc_intensity_audit_flags.csv,2


## Arousal Trajectories

The report-facing trajectory figure uses three equal-width panels: ADHD, Autism, and comparator terms. The ADHD and Autism panels foreground the substantive-core Overall trajectory and add clinical/disorder and lived-experience traces as lighter contextual lines.

Mixed-frame estimates, coverage diagnostics, small-cell warnings, collocate concentration, and trend diagnostics remain available in the saved CSV tables and audit flags. They are not saved as separate report figures in order to keep the figure folder aligned with the main dissertation story.


In [7]:
READER_FRAME_STRATA = ["clinical_only", "lived_only"]
READER_FRAME_LABELS = {
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
}


def save_lsc_figure(fig: plt.Figure, png_path: Path) -> Path:
    fig.tight_layout(pad=1.1, rect=[0, 0, 1, 0.93])
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    return pdf_path


def series_color(unit: str, frame_stratum: str) -> str:
    if unit in CONDITION_FRAME_COLORS and frame_stratum in CONDITION_FRAME_COLORS[unit]:
        return CONDITION_FRAME_COLORS[unit][frame_stratum]
    if unit in UNIT_COLOURS:
        return UNIT_COLOURS[unit]
    return FRAME_COLORS.get(frame_stratum, "#7B8785")


def trend_line_for(series: pd.DataFrame, trend: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    years = series["lsc_year"].to_numpy(dtype=float)
    fitted = trend["linear_intercept"] + trend["linear_slope_per_year"] * (years - trend["year_center"])
    return years, fitted


def trend_for(unit: str, frame_stratum: str) -> pd.Series | None:
    row = trend_summary.loc[
        trend_summary["analysis_unit"].eq(unit) & trend_summary["frame_stratum"].eq(frame_stratum)
    ]
    if row.empty or pd.isna(row.iloc[0]["linear_slope_per_year"]):
        return None
    return row.iloc[0]


def y_limits_from(frame: pd.DataFrame, value_column: str, ci_low: str | None = None, ci_high: str | None = None) -> tuple[float, float]:
    values = [frame[value_column].to_numpy(dtype=float)]
    if ci_low and ci_low in frame:
        values.append(frame[ci_low].to_numpy(dtype=float))
    if ci_high and ci_high in frame:
        values.append(frame[ci_high].to_numpy(dtype=float))
    finite_values = [v[np.isfinite(v)] for v in values if len(v)]
    combined = np.concatenate(finite_values)
    low, high = float(combined.min()), float(combined.max())
    padding = max((high - low) * 0.10, 0.004)
    return low - padding, high + padding


def style_year_axis(ax: plt.Axes) -> None:
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.tick_params(axis="x", labelsize=8.5)


def plot_line_with_trend(
    ax: plt.Axes,
    frame: pd.DataFrame,
    unit: str,
    frame_stratum: str,
    value_column: str,
    color: str,
    marker: str,
    label: str | None = None,
    ci_low: str | None = None,
    ci_high: str | None = None,
    ribbon_alpha: float = 0.12,
    linewidth: float = 2.2,
    markersize: float = 4.8,
    alpha: float = 1.0,
    show_trend: bool = True,
) -> None:
    series = frame.loc[frame["analysis_unit"].eq(unit) & frame["frame_stratum"].eq(frame_stratum)].sort_values("lsc_year")
    if series.empty:
        return
    ax.plot(
        series["lsc_year"],
        series[value_column],
        marker=marker,
        markersize=markersize,
        linewidth=linewidth,
        label=label,
        color=color,
        alpha=alpha,
    )
    if ci_low and ci_high:
        ax.fill_between(
            series["lsc_year"].to_numpy(dtype=float),
            series[ci_low].to_numpy(dtype=float),
            series[ci_high].to_numpy(dtype=float),
            color=color,
            alpha=ribbon_alpha,
            linewidth=0,
        )
    trend = trend_for(unit, frame_stratum)
    if show_trend and trend is not None:
        years, fitted = trend_line_for(series, trend)
        ax.plot(years, fitted, color=color, linewidth=1.05, linestyle="--", alpha=min(alpha + 0.12, 0.92))


def plot_target_panel(ax: plt.Axes, unit: str) -> None:
    plot_line_with_trend(
        ax,
        annual_arousal,
        unit,
        "substantive_core_overall",
        "arousal_mean",
        series_color(unit, "substantive_core_overall"),
        FRAME_MARKERS["substantive_core_overall"],
        "Overall",
        "arousal_ci_low",
        "arousal_ci_high",
        ribbon_alpha=0.14,
        linewidth=2.8,
        markersize=4.8,
    )
    for frame_stratum in READER_FRAME_STRATA:
        plot_line_with_trend(
            ax,
            annual_arousal,
            unit,
            frame_stratum,
            "arousal_mean",
            series_color(unit, frame_stratum),
            FRAME_MARKERS[frame_stratum],
            READER_FRAME_LABELS[frame_stratum],
            "arousal_ci_low",
            "arousal_ci_high",
            ribbon_alpha=0.075,
            linewidth=1.55,
            markersize=3.7,
            alpha=0.82,
        )
    ax.set_title(unit, loc="left", fontsize=11, fontweight="bold")
    style_year_axis(ax)
    ax.legend(loc="best", fontsize=7.6)


def plot_baseline_panel(
    ax: plt.Axes,
    frame: pd.DataFrame,
    value_column: str,
    ci_low: str | None = None,
    ci_high: str | None = None,
    show_trend: bool = True,
) -> None:
    for unit in BASELINE_UNITS:
        plot_line_with_trend(
            ax,
            frame,
            unit,
            BASELINE_FRAME_STRATUM,
            value_column,
            UNIT_COLOURS[unit],
            UNIT_MARKERS[unit] if "UNIT_MARKERS" in globals() else LSC_UNIT_MARKERS[unit],
            LSC_UNIT_LABELS[unit] if "LSC_UNIT_LABELS" in globals() else unit,
            ci_low,
            ci_high,
            ribbon_alpha=0.08,
            linewidth=2.0,
            show_trend=show_trend,
        )
    ax.set_title("Comparator terms", loc="left", fontsize=11, fontweight="bold")
    ax.legend(loc="best", fontsize=7.6)
    style_year_axis(ax)


main_rows = pd.concat(
    [
        annual_arousal.loc[
            annual_arousal["analysis_unit"].isin(TARGET_UNITS)
            & annual_arousal["frame_stratum"].isin(["substantive_core_overall", *READER_FRAME_STRATA])
        ],
        annual_arousal.loc[annual_arousal["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    ],
    ignore_index=True,
)
main_ylim = y_limits_from(main_rows, "arousal_mean", "arousal_ci_low", "arousal_ci_high")

fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.35), sharex=True, sharey=True)
fig.suptitle("Intensity: arousal near target terms", fontsize=14, fontweight="bold", x=0.02, ha="left")
for ax, unit in zip(axes[:2], TARGET_UNITS):
    plot_target_panel(ax, unit)
plot_baseline_panel(
    axes[2],
    annual_arousal.loc[annual_arousal["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    "arousal_mean",
    "arousal_ci_low",
    "arousal_ci_high",
)
for ax in axes:
    ax.set_ylim(*main_ylim)
    ax.set_xlabel("Publication year")
axes[0].set_ylabel("Mean arousal (-1 to 1)")
trajectory_pdf = save_lsc_figure(fig, TRAJECTORY_PLOT_PATH)
plt.close(fig)



## Coverage Overview

Coverage is high when most contexts produce at least one NRC-VAD match and when a large share of candidate collocate token positions are matched. These diagnostics should be checked before interpreting sharp frame-specific movements.


In [8]:
if "audit_flags" not in coverage_for_flags.columns:
    coverage_for_flags = coverage_for_flags.assign(audit_flags="")

coverage_summary = (
    coverage_for_flags.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        min_context_match_coverage=("context_match_coverage", "min"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        years_with_audit_flags=("audit_flags", lambda values: int(values.astype(str).ne("").sum())),
    )
    .sort_values(["analysis_unit", "frame_stratum"])
)

print("Coverage diagnostics are saved as CSV rather than a report figure:")
print(f"- {COVERAGE_PATH.relative_to(PROJECT_ROOT)}")
display(coverage_summary.round(3))


Coverage diagnostics are saved as CSV rather than a report figure:
- data/processed/lsc/intensity/lsc_intensity_coverage.csv


,analysis_unit,frame_stratum,min_context_match_coverage,min_matched_token_coverage,years_with_audit_flags
0,ADHD,clinical_only,0.962,0.888,0
1,ADHD,lived_only,0.954,0.864,0
2,ADHD,mixed,0.967,0.863,0
3,ADHD,substantive_core_overall,0.962,0.887,0
4,Autism,clinical_only,0.992,0.880,0
5,Autism,lived_only,0.986,0.866,0
6,Autism,mixed,0.967,0.832,0
7,Autism,substantive_core_overall,0.989,0.876,0
8,frustration,unframed_baseline,0.988,0.902,0
9,loneliness,unframed_baseline,0.993,0.891,0


## Handoff Summary

The handoff summary reports the mean, sample standard deviation, and range of the annual arousal estimates alongside coverage, warning counts, and trend slopes. The descriptive mean and standard deviation summarise annual trajectory values rather than collocate- or context-level observations.


In [9]:
summary = (
    annual_arousal.groupby(["analysis_unit", "frame_stratum"], as_index=False)
    .agg(
        years=("lsc_year", "nunique"),
        arousal_annual_mean=("arousal_mean", "mean"),
        arousal_annual_sd=("arousal_mean", "std"),
        arousal_min=("arousal_mean", "min"),
        arousal_max=("arousal_mean", "max"),
        matched_units=("matched_vad_units", "sum"),
        documents_with_matches=("documents_with_matches", "sum"),
        min_context_match_coverage=("context_match_coverage", "min"),
        min_matched_token_coverage=("matched_token_coverage", "min"),
        small_cell_years=("small_cell_flag", "sum"),
    )
)
warning_counts = (
    audit_flags.groupby(["analysis_unit", "frame_stratum"]).size().rename("warnings").reset_index()
    if not audit_flags.empty
    else pd.DataFrame({"analysis_unit": [], "frame_stratum": [], "warnings": []})
)
summary = summary.merge(warning_counts, on=["analysis_unit", "frame_stratum"], how="left").fillna({"warnings": 0})
summary = summary.merge(
    trend_summary[["analysis_unit", "frame_stratum", "linear_slope_per_year", "linear_p_value", "autocorrelation_flag"]],
    on=["analysis_unit", "frame_stratum"],
    how="left",
)
summary["warnings"] = summary["warnings"].astype(int)
summary["small_cell_years"] = summary["small_cell_years"].astype(int)
expected_rows = len(EXPECTED_YEARS) * (len(BASELINE_UNITS) + len(TARGET_UNITS) * len(TARGET_FRAME_STRATA))
if len(annual_arousal) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} annual rows, found {len(annual_arousal)}.")
summary


,analysis_unit,frame_stratum,years,arousal_annual_mean,arousal_annual_sd,arousal_min,arousal_max,matched_units,documents_with_matches,min_context_match_coverage,min_matched_token_coverage,small_cell_years,warnings,linear_slope_per_year,linear_p_value,autocorrelation_flag
0,ADHD,clinical_only,13,0.031835,0.008044,0.015353,0.040571,40772,7903,0.961938,0.888416,0,0,0.000212,0.738479,False
1,ADHD,lived_only,13,0.000273,0.012959,-0.024568,0.023217,11246,2702,0.953704,0.864426,0,0,0.001496,0.123193,False
2,ADHD,mixed,13,0.006381,0.017161,-0.021090,0.038490,6648,1609,0.966667,0.863436,1,1,-0.000152,0.910699,False
3,ADHD,substantive_core_overall,13,0.022916,0.006545,0.010810,0.032091,58666,11319,0.962389,0.887415,0,0,0.000065,0.899570,False
4,Autism,clinical_only,13,0.030247,0.006889,0.016605,0.040379,78302,12932,0.991758,0.879640,0,0,0.000986,0.047890,False
5,Autism,lived_only,13,-0.016087,0.028871,-0.030799,0.079337,43105,8995,0.986175,0.865809,0,0,0.002541,0.251506,False
6,Autism,mixed,13,-0.008827,0.012621,-0.035092,0.010355,23261,4713,0.966667,0.832168,0,0,0.000427,0.667879,False
7,Autism,substantive_core_overall,13,0.009681,0.011722,-0.000419,0.046970,144668,24049,0.989362,0.876249,0,0,0.001092,0.222925,False
8,frustration,unframed_baseline,13,0.020226,0.007441,0.000857,0.028772,360766,93157,0.988223,0.901552,0,1,-0.000443,0.446137,True
9,loneliness,unframed_baseline,13,0.022621,0.007822,0.011941,0.036289,131472,30331,0.992711,0.891103,0,0,-0.000604,0.318524,False
